# Multilingual RAG retrieval

**What you will learn**

- Index a bilingual corpus (English + Hindi) in a vector database
- Search with optional **language metadata filters** (`en`, `hi`, or both)
- See cross-lingual retrieval when no filter is applied

**Corpus:** [Universal Declaration of Human Rights](https://www.un.org/en/about-us/universal-declaration-of-human-rights) (UDHR) — preamble + articles in `data/udhr_en.txt` and `data/udhr_hi.txt` (same document, translated).

**Stack:** [multilingual-e5-base](https://huggingface.co/intfloat/multilingual-e5-base) embeddings + [LangChain Chroma](https://python.langchain.com/docs/integrations/vectorstores/chroma/) vector database.

**Prerequisites** (from repo root):

```bash
uv sync
uv run jupyter lab experiments/multilingual-rag-retrieval/notebook.ipynb
```

Run all cells **top to bottom**.

**First run:** The embedding model [`intfloat/multilingual-e5-base`](https://huggingface.co/intfloat/multilingual-e5-base) is downloaded from Hugging Face the first time `SentenceTransformer` loads (~**1.1 GB** on disk; size varies slightly by format). It is cached under `~/.cache/huggingface/hub/` by default; later runs reuse the cache. Ingest may take **several minutes** the first time (download + embedding ~122 chunks on CPU). Subsequent runs are faster if the model cache and local `vector_db/` folder already exist.

## Concepts (quick links)

| Idea | Link |
|------|------|
| RAG (retrieve, then optionally generate) | [LangChain RAG](https://python.langchain.com/docs/concepts/rag/) |
| Embeddings / semantic search | [Getting started with embeddings](https://huggingface.co/blog/getting-started-with-embeddings) |
| E5 model (`query:` / `passage:` prefixes) | [multilingual-e5-base](https://huggingface.co/intfloat/multilingual-e5-base) |
| Vector database (this notebook uses Chroma) | [Chroma docs](https://docs.trychroma.com/) |
| Metadata filtering at query time | [Chroma where filters](https://docs.trychroma.com/reference/where-filter) |

**Flow:** load text → split by article → embed chunks → store in vector DB → search by query embedding. Optional `lang` filter limits which stored chunks are considered.

## 1. Locate data and index folders

Resolve where this experiment lives on disk (`EXPERIMENT_DIR`), where the UDHR text files live (`DATA_DIR`), and where the on-disk vector index will be written (`vector_db/`). The index folder is local Chroma storage and is gitignored — you rebuild it when you run ingest.

In [1]:
from pathlib import Path

# Jupyter cwd may be repo root or this experiment folder — detect via notebook.ipynb.
EXPERIMENT_DIR = Path.cwd().resolve()
if not (EXPERIMENT_DIR / "notebook.ipynb").exists():
    alt = Path("experiments/multilingual-rag-retrieval").resolve()
    if (alt / "notebook.ipynb").exists():
        EXPERIMENT_DIR = alt

DATA_DIR = EXPERIMENT_DIR / "data"
VECTOR_DB_PATH = EXPERIMENT_DIR / "vector_db"  # on-disk index (Chroma backend)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Experiment dir: {EXPERIMENT_DIR}")
print(f"Vector DB path: {VECTOR_DB_PATH}")

Experiment dir: /Users/ysskrishna/siva/Projects/awesome-rag-experiments/experiments/multilingual-rag-retrieval
Vector DB path: /Users/ysskrishna/siva/Projects/awesome-rag-experiments/experiments/multilingual-rag-retrieval/vector_db


## 2. Define helpers — E5 embeddings, article chunking, index builder

This cell **defines** functions and classes only; it does not load the corpus or build the index yet. You will run ingest in the next section. Helpers cover: loading UDHR text, splitting on article headings with shared `article` metadata, wrapping the E5 model with `passage:` / `query:` prefixes, and creating the Chroma-backed `vector_db`.

In [2]:
import re
import shutil

import pandas as pd
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

COLLECTION_NAME = "multilingual_rag_facts"
MODEL_NAME = "intfloat/multilingual-e5-base"

LANG_SOURCE_MAP = {
    "en": "udhr_en.txt",
    "hi": "udhr_hi.txt",
}

# Split UDHR text on article headings (English vs Hindi patterns).
HI_ARTICLE_NUM = r"[०-९\d]+"
ARTICLE_MARKERS = {
    "en": re.compile(r"(?=\bArticle\s+(\d+)\b)", re.IGNORECASE),
    "hi": re.compile(rf"(?=अनुच्छेद\s*({HI_ARTICLE_NUM})\.?)"),
}
ARTICLE_ID_FROM_START = {
    "en": re.compile(r"^\s*Article\s+(\d+)\b", re.IGNORECASE),
    "hi": re.compile(rf"^\s*अनुच्छेद\s*({HI_ARTICLE_NUM})\.?"),
}
# Normalize Devanagari digits so article "३" and "3" share the same id.
DEVANAGARI_DIGITS = str.maketrans("०१२३४५६७८९", "0123456789")


class E5Embeddings(Embeddings):
    """E5 expects 'passage:' for indexed chunks and 'query:' for search queries."""

    def __init__(self, model_name: str = MODEL_NAME) -> None:
        # First run downloads from Hugging Face Hub (~1.1 GB); later runs use cache.
        self._model = SentenceTransformer(model_name)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        prefixed = [f"passage: {t}" for t in texts]
        vectors = self._model.encode(
            prefixed, normalize_embeddings=True, show_progress_bar=True
        )
        return vectors.tolist()

    def embed_query(self, text: str) -> list[float]:
        vectors = self._model.encode(
            [f"query: {text}"], normalize_embeddings=True, show_progress_bar=False
        )
        return vectors[0].tolist()


def _normalize_article_num(raw: str) -> str:
    return raw.translate(DEVANAGARI_DIGITS)


def _article_id(lang: str, chunk: str) -> str:
    pattern = ARTICLE_ID_FROM_START.get(lang)
    if pattern:
        match = pattern.search(chunk.strip())
        if match:
            return _normalize_article_num(match.group(1))
    return "preamble"


def load_lang_text(lang: str) -> str:
    path = DATA_DIR / LANG_SOURCE_MAP[lang]
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}")
    return path.read_text(encoding="utf-8")


def split_by_articles(lang: str, text: str) -> list[Document]:
    marker = ARTICLE_MARKERS.get(lang)
    if marker:
        parts = marker.split(text)
        parts = [p.strip() for p in parts if p.strip()]
        if len(parts) > 1:
            docs = []
            for part in parts:
                article = _article_id(lang, part)
                # article id is normalized so en "3" and hi "३" align across translations
                docs.append(
                    Document(
                        page_content=part,
                        metadata={"lang": lang, "article": article, "source": "udhr"},
                    )
                )
            return docs

    # Fallback if article markers are not found.
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
    return splitter.create_documents(
        [text],
        metadatas=[{"lang": lang, "article": "unknown", "source": "udhr"}],
    )


def build_documents(langs: list[str]) -> list[Document]:
    all_docs: list[Document] = []
    for lang in langs:
        text = load_lang_text(lang)
        if not text.strip():
            print(f"{lang}: 0 chars extracted — check source file")
        chunks = split_by_articles(lang, text)
        print(f"{lang}: {len(chunks)} chunks")
        all_docs.extend(chunks)
    return all_docs


def build_vector_db(documents: list[Document], *, reset: bool = True) -> Chroma:
    """Create or replace the on-disk vector index (Chroma backend)."""
    if reset and VECTOR_DB_PATH.exists():
        shutil.rmtree(VECTOR_DB_PATH)  # clean re-run of ingest
    VECTOR_DB_PATH.mkdir(parents=True, exist_ok=True)
    embeddings = E5Embeddings()
    return Chroma.from_documents(
        documents=documents,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
    )

## 3. Ingest — chunk, embed, and persist the bilingual index

Loads English and Hindi UDHR files, splits them into ~61 chunks per language (~122 total), embeds each chunk, and writes them to `vector_db/`. Expect printed chunk counts, then progress bars for model load and encoding.

**First run reminder:** If you have not used this model before, expect a large Hugging Face download when `SentenceTransformer` loads inside `build_vector_db`.

In [3]:
langs = ["en", "hi"]

documents = build_documents(langs)
vector_db = build_vector_db(documents, reset=True)
print(f"Indexed {len(documents)} chunks into '{COLLECTION_NAME}'")

en: 61 chunks
hi: 61 chunks


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Indexed 122 chunks into 'multilingual_rag_facts'


## 4. Search helper — similarity search with optional lang filter

`search_udhr` embeds your query (with the E5 `query:` prefix), runs similarity search against the index, and returns a small table (rank, article, lang, score, text preview).

- `lang_filter="en"` or `"hi"` → only chunks with that metadata `lang`
- `lang_filter=None` → search the full bilingual index (cross-lingual)
- **Score:** Chroma returns distance; **lower = closer** match

In [4]:
def search_udhr(
    vector_db: Chroma,
    query: str,
    *,
    k: int = 5,
    lang_filter: str | None = None,
) -> pd.DataFrame:
    """Similarity search with optional metadata filter on lang (en | hi)."""
    metadata_filter = {"lang": lang_filter} if lang_filter else None
    hits = vector_db.similarity_search_with_score(
        query, k=k, filter=metadata_filter
    )
    rows = []
    for rank, (doc, score) in enumerate(hits, start=1):
        text = doc.page_content
        preview = text[:120] + "..." if len(text) > 120 else text
        rows.append(
            {
                "rank": rank,
                "article": doc.metadata.get("article"),
                "lang": doc.metadata.get("lang"),
                "score": round(float(score), 4),
                "text": preview,
            }
        )
    return pd.DataFrame(rows)

## 5. Three search cases

We use **Article 3** (right to life, liberty, security) — the same article in both language files. Queries below are paraphrases of that article. The `assert` checks encode **expected retrieval for this corpus and model**, not a guarantee for every RAG system.

| Case | `lang_filter` | Expected |
|------|---------------|----------|
| A | `"en"` | All results `lang=en`; **rank 1 = article 3** with English Article 3 text |
| B | `"hi"` | All results `lang=hi`; **rank 1 = article 3** with Hindi Article 3 text |
| C | `None` | Mix of `en` and `hi`; **article 3 appears in both languages** in top 10 |

In [5]:
# Ground truth for retrieval checks (from data/udhr_en.txt and data/udhr_hi.txt, Article 3)
EXPECTED_ARTICLE = "3"
ARTICLE_3_EN_SNIPPET = "right to life, liberty"
ARTICLE_3_HI_SNIPPET = "जीवन, स्वाधीनता"

### Case A — English only (`lang_filter="en"`)

English paraphrase of Article 3. You should see only `lang=en`, with **rank 1** pointing at **article 3** and the life/liberty/security passage.

In [6]:
# Paraphrase of UDHR Article 3 — rank 1 should retrieve that article in English.
query_en = "Everyone has the right to life, liberty and security of person."

df_en = search_udhr(vector_db, query_en, k=5, lang_filter="en")
display(df_en)
print("Languages in results:", df_en["lang"].value_counts().to_dict())
assert set(df_en["lang"]) <= {"en"}, "Expected only English chunks"
assert df_en.iloc[0]["article"] == EXPECTED_ARTICLE, "Expected Article 3 at rank 1"
assert ARTICLE_3_EN_SNIPPET in df_en.iloc[0]["text"], "Expected Article 3 English text"

,rank,article,lang,score,text
0,1,3,en,0.2505,"Article 3\nEveryone has the right to life, lib..."
1,2,2,en,0.2835,Article 2\nEveryone is entitled to all the rig...
2,3,22,en,0.2852,"Article 22\nEveryone, as a member of society, ..."
3,4,25,en,0.2954,Article 25\n 1. Everyone has the right to a st...
4,5,13,en,0.3084,Article 13\n 1. Everyone has the right to free...


Languages in results: {'en': 5}


### Case B — Hindi only (`lang_filter="hi"`)

Hindi paraphrase of Article 3. You should see only `lang=hi`, with **rank 1** at **article 3** and matching Hindi text.

In [7]:
# Paraphrase of UDHR Article 3 — rank 1 should retrieve that article in Hindi.
query_hi = "प्रत्येक व्यक्ति को जीवन, स्वाधीनता और वैयक्तिक सुरक्षा का अधिकार है।"

df_hi = search_udhr(vector_db, query_hi, k=5, lang_filter="hi")
display(df_hi)
print("Languages in results:", df_hi["lang"].value_counts().to_dict())
assert set(df_hi["lang"]) <= {"hi"}, "Expected only Hindi chunks"
assert df_hi.iloc[0]["article"] == EXPECTED_ARTICLE, "Expected Article 3 at rank 1"
assert ARTICLE_3_HI_SNIPPET in df_hi.iloc[0]["text"], "Expected Article 3 Hindi text"

,rank,article,lang,score,text
0,1,3,hi,0.1587,"अनुच्छेद ३.\nप्रत्येक व्यक्ति को जीवन, स्वाधीन..."
1,2,22,hi,0.2902,अनुच्छेद २२.\nसमाज के एक सदस्य के रूप में प्रत...
2,3,28,hi,0.3191,अनुच्छेद २८.\nप्रत्येक व्यक्ति को ऐसी सामाजिक ...
3,4,6,hi,0.3256,अनुच्छेद ६.\nहर किसी को हर जगह क़ानून की निग़ा...
4,5,13,hi,0.3275,अनुच्छेद १३.\n(१) प्रत्येक व्यक्ति को प्रत्येक...


Languages in results: {'hi': 5}


### Case C — No filter (English + Hindi)

Same English query as Case A, but search the **full** index with `k=10`. You should see both languages; **article 3** should appear once in English and once in Hindi among the top hits.

In [8]:
# Same Article 3 query as Case A, but no lang filter — cross-lingual top hits.
df_both = search_udhr(vector_db, query_en, k=10, lang_filter=None)
display(df_both)
print("Languages in results:", df_both["lang"].value_counts().to_dict())
assert {"en", "hi"} <= set(df_both["lang"]), "Expected both English and Hindi chunks"
article_3_langs = set(df_both.loc[df_both["article"] == EXPECTED_ARTICLE, "lang"])
assert article_3_langs == {"en", "hi"}, "Expected Article 3 in both languages in top 10"

,rank,article,lang,score,text
0,1,3,hi,0.1657,"अनुच्छेद ३.\nप्रत्येक व्यक्ति को जीवन, स्वाधीन..."
1,2,3,en,0.2505,"Article 3\nEveryone has the right to life, lib..."
2,3,6,hi,0.2759,अनुच्छेद ६.\nहर किसी को हर जगह क़ानून की निग़ा...
3,4,2,en,0.2835,Article 2\nEveryone is entitled to all the rig...
4,5,22,en,0.2852,"Article 22\nEveryone, as a member of society, ..."
5,6,22,hi,0.2890,अनुच्छेद २२.\nसमाज के एक सदस्य के रूप में प्रत...
6,7,28,hi,0.2925,अनुच्छेद २८.\nप्रत्येक व्यक्ति को ऐसी सामाजिक ...
7,8,25,en,0.2954,Article 25\n 1. Everyone has the right to a st...
8,9,13,en,0.3084,Article 13\n 1. Everyone has the right to free...
9,10,18,en,0.3086,Article 18\nEveryone has the right to freedom ...


Languages in results: {'en': 6, 'hi': 4}


## Wrap-up

- **`lang_filter`** restricts which **stored chunks** are searched, not the language you type in the query.
- **No filter** lets multilingual embeddings retrieve the best matches across the whole index — useful when the same facts exist in multiple languages.
- **Next step:** pass retrieved chunks to an LLM to generate an answer (full RAG pipeline).